# 10 · RNA velocity of oligodendrocyte maturation (scVelo) — GSE216146

**Kernel: `velocity`** (env `envs/velocity.yaml`). scVelo 0.3 fails with pandas 3 / numpy 2.5 of the main env, so
this notebook runs in a separate environment and only reads the file exported by `09_pseudotime`.

Input: newly formed + mature oligodendrocytes with the **authors'** `spliced` / `unspliced` counts (velocyto; the only
source of splicing information, user decision 2026-11-13), our QC, Harmony PCs, lineage UMAP and DPT pseudotime.

- **Model:** scVelo **stochastic** model (`velocity.mode`). The dynamical model was also fitted; it reverses the known
  direction (newly formed cells come out as the end point), a known failure mode in near-steady-state populations,
  so it is shown only as a check.
- **Validation:** agreement of velocity pseudotime with DPT (09), terminal states per stage, per batch.
- **Comparison:** per sample median velocity pseudotime and velocity confidence, OLS `~ batch + condition`
  (n = 3 vs 3, descriptive). The unspliced fraction is higher in batch D20 (~44 % vs ~31 %), so batch is in the model.

**Input:** `results/09_lineage_velocity_input.h5ad` · **Output:** `results/10_velocity_obs.tsv.gz`,
tables `results/tables/10_*`, plots `results/plots/10_*`.

In [ ]:
import gc
import sys
import warnings
from contextlib import contextmanager
from importlib.metadata import version
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import sc_utils as scu

@contextmanager
def quiet():
    """scvelo 0.3 emits numpy DeprecationWarnings on every gene fit; silence them inside the calls."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        yield

cfg = scu.load_config(ROOT / "config.yaml")
p = cfg["velocity"]
seed = cfg["integration"]["random_state"]
scu.setup_plotting()
scv.settings.verbosity = 1
print("scvelo", version("scvelo"), "| scanpy", version("scanpy"), "| pandas", pd.__version__, "| numpy", np.__version__)
p

In [ ]:
adata = ad.read_h5ad(cfg["paths"]["results_dir"] / cfg["pseudotime"]["velocity_input"])
assert np.issubdtype(adata.layers["spliced"].dtype, np.integer), "layers must be raw integer counts"
s, u = adata.layers["spliced"].sum(axis=1).A1, adata.layers["unspliced"].sum(axis=1).A1
adata.obs["unspliced_frac"] = u / (u + s)
qc = adata.obs.groupby(["biosample", "condition", "batch"], observed=True)["unspliced_frac"].median().rename("median_unspliced_frac")
qc.round(3).to_frame()

## 1. Preprocessing and stochastic velocity

In [ ]:
scv.pp.filter_and_normalize(adata, min_shared_counts=p["min_shared_counts"])
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=p["n_top_genes"], batch_key="biosample")
adata = adata[:, adata.var["highly_variable"]].copy()
sc.pp.neighbors(adata, n_neighbors=p["n_neighbors"], use_rep="X_pca_harmony", random_state=seed)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

with quiet():
    scv.tl.velocity(adata, mode=p["mode"])
    scv.tl.velocity_graph(adata, n_jobs=p["n_jobs"], show_progress_bar=False)
    scv.tl.velocity_confidence(adata)
    scv.tl.terminal_states(adata)
    scv.tl.velocity_pseudotime(adata)
adata

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
scv.pl.velocity_embedding_stream(adata, basis="umap", color="stage", ax=axes[0], show=False, legend_loc="right margin",
                                 title="Stochastic velocity", size=25, alpha=0.6)
scv.pl.scatter(adata, basis="umap", color="velocity_pseudotime", cmap="gnuplot", ax=axes[1], show=False,
               title="velocity pseudotime", size=25)
scv.pl.scatter(adata, basis="umap", color="dpt_pseudotime", cmap="gnuplot", ax=axes[2], show=False,
               title="DPT (09)", size=25)
fig.tight_layout()
scu.save_fig(fig, cfg, "10_velocity_stream")
plt.show()

In [ ]:
# Validation: direction against DPT and known stage order, overall and per batch
obs = adata.obs
check = pd.DataFrame({
    "rho_vs_dpt": [obs["velocity_pseudotime"].corr(obs["dpt_pseudotime"], method="spearman")] +
                  [obs.loc[obs["batch"] == b, "velocity_pseudotime"].corr(obs.loc[obs["batch"] == b, "dpt_pseudotime"],
                                                                      method="spearman") for b in ["D20", "D21"]],
}, index=["all", "D20", "D21"])
display(check.round(3))
by_stage = obs.groupby("stage", observed=True)[["velocity_pseudotime", "dpt_pseudotime", "root_cells", "end_points",
                                                 "velocity_confidence"]].median()
by_stage.round(3).to_csv(cfg["paths"]["tables_dir"] / "10_velocity_by_stage.tsv", sep="\t")
by_stage.round(3)

## 2. Check: dynamical model

In [ ]:
if p["check_dynamical"]:
    dyn = adata.copy()
    dyn.obs = dyn.obs.drop(columns=["root_cells", "end_points", "velocity_pseudotime"])   # own roots, not the stochastic ones
    with quiet():
        scv.tl.recover_dynamics(dyn, n_jobs=p["n_jobs"], show_progress_bar=False)
        scv.tl.velocity(dyn, mode="dynamical")
        scv.tl.velocity_graph(dyn, n_jobs=p["n_jobs"], show_progress_bar=False)
        scv.tl.terminal_states(dyn)
        scv.tl.velocity_pseudotime(dyn)
        scv.tl.latent_time(dyn)
    adata.obs["dynamical_velocity_pseudotime"] = dyn.obs["velocity_pseudotime"]
    adata.obs["dynamical_latent_time"] = dyn.obs["latent_time"]
    print("dynamical: rho(velocity pseudotime, DPT) =",
          round(dyn.obs["velocity_pseudotime"].corr(dyn.obs["dpt_pseudotime"], method="spearman"), 3))
    display(dyn.obs.groupby("stage", observed=True)[["velocity_pseudotime", "latent_time"]].median().round(3))
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    scv.pl.velocity_embedding_stream(dyn, basis="umap", color="stage", ax=ax, show=False, legend_loc="none",
                                     title="Dynamical model (check: direction reversed)", size=25, alpha=0.6)
    scu.save_fig(fig, cfg, "10_velocity_stream_dynamical")
    plt.show()
    del dyn
    gc.collect()

## 3. Chemotherapy vs control

In [ ]:
meta = (obs[["biosample", "condition", "batch"]].drop_duplicates().set_index("biosample")
        .sort_values(["condition", "batch"]))
meta.index = meta.index.astype(str)
meta["condition"] = meta["condition"].cat.remove_unused_categories()
meta["batch"] = meta["batch"].astype(str)
per_sample = obs.groupby("biosample", observed=True).agg(
    median_velocity_pseudotime=("velocity_pseudotime", "median"),
    median_velocity_confidence=("velocity_confidence", "median"),
    median_unspliced_frac=("unspliced_frac", "median"))
per_sample.index = per_sample.index.astype(str)
test = scu.ols_by_column(per_sample, meta, p["formula"])
per_sample.join(meta).round(3).to_csv(cfg["paths"]["tables_dir"] / "10_per_sample.tsv", sep="\t")
test.to_csv(cfg["paths"]["tables_dir"] / "10_condition_test.tsv", sep="\t")
display(per_sample.join(meta).round(3))
test.round(4)

In [ ]:
# Streams per condition: neighbors, moments and velocity recomputed within each condition
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, cond in zip(axes, meta["condition"].cat.categories):
    sub = adata[adata.obs["condition"] == cond].copy()
    sc.pp.neighbors(sub, n_neighbors=p["n_neighbors"], use_rep="X_pca_harmony", random_state=seed)
    with quiet():
        scv.pp.moments(sub, n_pcs=None, n_neighbors=None)
        scv.tl.velocity(sub, mode=p["mode"])
        scv.tl.velocity_graph(sub, n_jobs=p["n_jobs"], show_progress_bar=False)
    scv.pl.velocity_embedding_stream(sub, basis="umap", color="stage", ax=ax, show=False, size=25, alpha=0.6,
                                     legend_loc="right margin" if ax is axes[-1] else "none",
                                     title=f"{cond} ({sub.n_obs} cells)")
    del sub
fig.tight_layout()
scu.save_fig(fig, cfg, "10_velocity_stream_by_condition")
plt.show()

In [ ]:
cols = ["biosample", "condition", "batch", "stage", "dpt_pseudotime", "velocity_pseudotime", "velocity_confidence",
        "unspliced_frac", "root_cells", "end_points"] + [c for c in adata.obs if c.startswith("dynamical_")]
adata.obs[cols].to_csv(cfg["paths"]["results_dir"] / "10_velocity_obs.tsv.gz", sep="\t")
del adata
_ = gc.collect()